<a href="https://colab.research.google.com/github/aditi1295/MachineLearning/blob/main/optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 22.8 MB/s eta 0:00:00


In [2]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [6]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2026-09-18 14:34:41,843] A new study created in memory with name: no-name-82a7f64f-1938-4bf3-aaca-f6e77580d74e
[I 2026-09-18 14:34:43,314] Trial 0 finished with value: 0.7802607076350093 and parameters: {'n_estimators': 126, 'max_depth': 16}. Best is trial 0 with value: 0.7802607076350093.
[I 2026-09-18 14:34:44,684] Trial 1 finished with value: 0.7783985102420857 and parameters: {'n_estimators': 124, 'max_depth': 17}. Best is trial 0 with value: 0.7802607076350093.
[I 2026-09-18 14:34:45,925] Trial 2 finished with value: 0.7783985102420857 and parameters: {'n_estimators': 55, 'max_depth': 17}. Best is trial 0 with value: 0.7802607076350093.
[I 2026-09-18 14:34:47,999] Trial 3 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 84, 'max_depth': 19}. Best is trial 0 with value: 0.7802607076350093.
[I 2026-09-18 14:34:49,112] Trial 4 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 85, 'max_depth': 5}. Best is trial 0 with value: 0.78026070

In [7]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 197, 'max_depth': 19}


In [8]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


#Samplers in Optuna

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [10]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-09-18 14:36:00,068] A new study created in memory with name: no-name-03a0581a-be7a-4aa3-8561-3231ca88bbdc
[I 2026-09-18 14:36:00,603] Trial 0 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 60, 'max_depth': 9}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-09-18 14:36:01,994] Trial 1 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 140, 'max_depth': 9}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-09-18 14:36:03,042] Trial 2 finished with value: 0.7560521415270017 and parameters: {'n_estimators': 107, 'max_depth': 3}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-09-18 14:36:05,870] Trial 3 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 176, 'max_depth': 5}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-09-18 14:36:06,572] Trial 4 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 53, 'max_depth': 16}. Best is trial 4 with value: 0.7728119180

In [11]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 194, 'max_depth': 17}


In [12]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.75


In [13]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [14]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-09-18 14:37:13,184] A new study created in memory with name: no-name-257f2bd3-081b-4402-b3e3-bf3b9aa609d6
[I 2026-09-18 14:37:14,055] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-09-18 14:37:15,632] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-09-18 14:37:16,244] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-09-18 14:37:17,345] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-09-18 14:37:18,477] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [15]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [16]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


#oputna visualization

In [17]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [18]:
# 1. Optimization History
plot_optimization_history(study).show()

In [19]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [20]:
# 3. Slice Plot
plot_slice(study).show()

In [21]:
# 4. Contour Plot
plot_contour(study).show()

In [22]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

#Optimizing Multiple ML Models

In [23]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [24]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [25]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-09-18 14:40:04,462] A new study created in memory with name: no-name-f0558af1-0abd-48a8-aeac-7d1abcbef929
[I 2026-09-18 14:40:04,493] Trial 0 finished with value: 0.7392923649906891 and parameters: {'classifier': 'SVM', 'C': 0.6753684914304938, 'kernel': 'poly', 'gamma': 'auto'}. Best is trial 0 with value: 0.7392923649906891.
[I 2026-09-18 14:40:04,566] Trial 1 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM', 'C': 12.652318502795119, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 1 with value: 0.7858472998137801.
[I 2026-09-18 14:40:04,606] Trial 2 finished with value: 0.7169459962756052 and parameters: {'classifier': 'SVM', 'C': 0.19196354403392138, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 1 with value: 0.7858472998137801.
[I 2026-09-18 14:40:04,654] Trial 3 finished with value: 0.696461824953445 and parameters: {'classifier': 'SVM', 'C': 88.68808120095807, 'kernel': 'sigmoid', 'gamma': 'scale'}. Best is trial 1 with value: 0.78584

In [26]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.12582675083427006, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [27]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,system_attrs_tpe:relative_params:0,state
0,0,0.739292,2026-09-18 14:40:04.463606,2026-09-18 14:40:04.493309,0 days 00:00:00.029703,0.675368,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.785847,2026-09-18 14:40:04.494889,2026-09-18 14:40:04.566532,0 days 00:00:00.071643,12.652319,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.716946,2026-09-18 14:40:04.567628,2026-09-18 14:40:04.606710,0 days 00:00:00.039082,0.191964,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.696462,2026-09-18 14:40:04.608215,2026-09-18 14:40:04.654008,0 days 00:00:00.045793,88.688081,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.737430,2026-09-18 14:40:04.655218,2026-09-18 14:40:07.129519,0 days 00:00:02.474301,NaN,NaN,GradientBoosting,NaN,NaN,0.030035,19.0,2.0,7.0,66.0,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.759777,2026-09-18 14:40:55.655568,2026-09-18 14:40:55.793617,0 days 00:00:00.138049,0.254394,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
96,96,0.787709,2026-09-18 14:40:55.800007,2026-09-18 14:40:55.907899,0 days 00:00:00.107892,0.156323,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
97,97,0.787709,2026-09-18 14:40:55.910972,2026-09-18 14:40:55.981035,0 days 00:00:00.070063,0.101992,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
98,98,0.785847,2026-09-18 14:40:55.985343,2026-09-18 14:40:56.077764,0 days 00:00:00.092421,0.198483,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE


In [28]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,80
GradientBoosting,10
RandomForest,10


In [29]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.738361
RandomForest,0.765363
SVM,0.775489


In [30]:
# 1. Optimization History
plot_optimization_history(study).show()

In [31]:
# 3. Slice Plot
plot_slice(study).show()

In [32]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [42]:
!pip install -U "optuna-integration[xgboost]"
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")
    # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 2.7 MB/s eta 0:00:00


[I 2026-09-18 14:49:10,681] A new study created in memory with name: no-name-e390facf-84f2-43a8-bbbd-8482fa1c5be8


[0]	train-mlogloss:0.95868	eval-mlogloss:0.95081
[1]	train-mlogloss:0.82619	eval-mlogloss:0.81177
[2]	train-mlogloss:0.71630	eval-mlogloss:0.69394
[3]	train-mlogloss:0.63170	eval-mlogloss:0.61056
[4]	train-mlogloss:0.55507	eval-mlogloss:0.53094
[5]	train-mlogloss:0.49597	eval-mlogloss:0.46842
[6]	train-mlogloss:0.44505	eval-mlogloss:0.41469
[7]	train-mlogloss:0.41083	eval-mlogloss:0.38179
[8]	train-mlogloss:0.37061	eval-mlogloss:0.33696
[9]	train-mlogloss:0.33396	eval-mlogloss:0.29685
[10]	train-mlogloss:0.30113	eval-mlogloss:0.26094
[11]	train-mlogloss:0.27327	eval-mlogloss:0.23301
[12]	train-mlogloss:0.25302	eval-mlogloss:0.21171
[13]	train-mlogloss:0.23464	eval-mlogloss:0.19445
[14]	train-mlogloss:0.21379	eval-mlogloss:0.17373
[15]	train-mlogloss:0.19713	eval-mlogloss:0.15504
[16]	train-mlogloss:0.18207	eval-mlogloss:0.13756
[17]	train-mlogloss:0.16859	eval-mlogloss:0.12319
[18]	train-mlogloss:0.15607	eval-mlogloss:0.11178
[19]	train-mlogloss:0.14721	eval-mlogloss:0.10429
[20]	train

/usr/lib/python3.13/importlib/__init__.py:88: FutureWarning:

`optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.



[62]	train-mlogloss:0.07606	eval-mlogloss:0.03720
[63]	train-mlogloss:0.07588	eval-mlogloss:0.03681
[64]	train-mlogloss:0.07575	eval-mlogloss:0.03696
[65]	train-mlogloss:0.07545	eval-mlogloss:0.03684
[66]	train-mlogloss:0.07519	eval-mlogloss:0.03561
[67]	train-mlogloss:0.07483	eval-mlogloss:0.03498
[68]	train-mlogloss:0.07484	eval-mlogloss:0.03508
[69]	train-mlogloss:0.07482	eval-mlogloss:0.03450
[70]	train-mlogloss:0.07448	eval-mlogloss:0.03517
[71]	train-mlogloss:0.07434	eval-mlogloss:0.03545
[72]	train-mlogloss:0.07370	eval-mlogloss:0.03426
[73]	train-mlogloss:0.07348	eval-mlogloss:0.03367
[74]	train-mlogloss:0.07349	eval-mlogloss:0.03347
[75]	train-mlogloss:0.07325	eval-mlogloss:0.03312
[76]	train-mlogloss:0.07303	eval-mlogloss:0.03310
[77]	train-mlogloss:0.07306	eval-mlogloss:0.03245
[78]	train-mlogloss:0.07316	eval-mlogloss:0.03213
[79]	train-mlogloss:0.07249	eval-mlogloss:0.03100
[80]	train-mlogloss:0.07232	eval-mlogloss:0.03084
[81]	train-mlogloss:0.07217	eval-mlogloss:0.03104


...

[0]	train-mlogloss:1.06727	eval-mlogloss:1.06900
[1]	train-mlogloss:1.03513	eval-mlogloss:1.03487
[2]	train-mlogloss:1.00527	eval-mlogloss:1.00365
[3]	train-mlogloss:0.97726	eval-mlogloss:0.97301
[4]	train-mlogloss:0.94925	eval-mlogloss:0.94341
[5]	train-mlogloss:0.92331	eval-mlogloss:0.91665
[6]	train-mlogloss:0.89836	eval-mlogloss:0.89040
[7]	train-mlogloss:0.87388	eval-mlogloss:0.86474
[8]	train-mlogloss:0.85030	eval-mlogloss:0.83961
[9]	train-mlogloss:0.82792	eval-mlogloss:0.81606
[10]	train-mlogloss:0.80594	eval-mlogloss:0.79346
[11]	train-mlogloss:0.78569	eval-mlogloss:0.77206
[12]	train-mlogloss:0.76619	eval-mlogloss:0.75222
[13]	train-mlogloss:0.74622	eval-mlogloss:0.73186
[14]	train-mlogloss:0.72752	eval-mlogloss:0.71246
[15]	train-mlogloss:0.70896	eval-mlogloss:0.69281
[16]	train-mlogloss:0.69176	eval-mlogloss:0.67349
[17]	train-mlogloss:0.67498	eval-mlogloss:0.65509
[18]	train-mlogloss:0.65736	eval-mlogloss:0.63744
[19]	train-mlogloss:0.64144	eval-mlogloss:0.62037
[20]	train

[I 2026-09-18 14:49:16,631] Trial 48 pruned. Trial was pruned at iteration 32.


[0]	train-mlogloss:1.07351	eval-mlogloss:1.07387
[1]	train-mlogloss:1.04797	eval-mlogloss:1.04671
[2]	train-mlogloss:1.02456	eval-mlogloss:1.02173
[3]	train-mlogloss:1.00183	eval-mlogloss:0.99665
[4]	train-mlogloss:0.97876	eval-mlogloss:0.97243
[5]	train-mlogloss:0.95740	eval-mlogloss:0.95063
[6]	train-mlogloss:0.93669	eval-mlogloss:0.92890
[7]	train-mlogloss:0.91697	eval-mlogloss:0.90821
[8]	train-mlogloss:0.89686	eval-mlogloss:0.88686
[9]	train-mlogloss:0.87770	eval-mlogloss:0.86638
[10]	train-mlogloss:0.85859	eval-mlogloss:0.84552
[11]	train-mlogloss:0.84157	eval-mlogloss:0.82717
[12]	train-mlogloss:0.82529	eval-mlogloss:0.81008
[13]	train-mlogloss:0.80796	eval-mlogloss:0.79179
[14]	train-mlogloss:0.79162	eval-mlogloss:0.77493
[15]	train-mlogloss:0.77488	eval-mlogloss:0.75745
[16]	train-mlogloss:0.75943	eval-mlogloss:0.74053
[17]	train-mlogloss:0.74443	eval-mlogloss:0.72434
[18]	train-mlogloss:0.72854	eval-mlogloss:0.70829
[19]	train-mlogloss:0.71431	eval-mlogloss:0.69310
[20]	train

[I 2026-09-18 14:49:16,801] Trial 49 pruned. Trial was pruned at iteration 32.


Best trial: {'lambda': 0.00432232676269759, 'alpha': 1.7219063672366305e-05, 'eta': 0.12831722446825466, 'gamma': 0.00015260127947542152, 'max_depth': 6, 'min_child_weight': 2, 'subsample': 0.6695314840050257, 'colsample_bytree': 0.7364141600688288}
Best accuracy: 1.0
